# Normalisierungs-Check: Negations-/Kritik-Kollaps

`normalize_stroemung` hatte einen Bug: Embedding-Aehnlichkeit kollabierte Negations-Komposita auf das Gegenteil (`antifeministisch` -> `feministisch` bei Distanz 0.18, `islamkritisch` -> `islamistisch` bei 0.09 — beide weit innerhalb der Match-Schwelle). Gefixt mit einem Marker-basierten Guard (`anti`, `kritisch`, `gegner`, `feindlich`, `skeptisch`, `ablehnend`) — siehe [ADR 0009](../docs/concepts/decisions/0009-pipeline-hardening-after-gemini-meta-review.md).

**Nie geprueft:** ob `normalize_technique` (embedding-basiert, wie stroemung_store) oder `normalize_role` (difflib-basiert, andere Methode) denselben latenten Bug haben. Dieses Notebook testet systematisch alle drei Normalizer gegen Negations-/Kritik-Varianten ihrer kanonischen Namen.

**Voraussetzung:** laufende ChromaDB fuer `normalize_technique`/`normalize_stroemung` (`normalize_role` braucht keine ChromaDB — reines difflib, siehe `role_store.py`).

## 1 — Setup

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from news_analyser.repositories.technique_store import normalize_technique, _TECHNIQUES
from news_analyser.repositories.role_store import normalize_role, _CANONICAL_NAMES as ROLE_NAMES
from news_analyser.repositories.stroemung_store import normalize_stroemung, _CANONICAL_NAMES as STROEMUNG_NAMES

# Dieselben Marker wie stroemung_store._is_oppositional — hier zum Abgleich,
# ob ein Normalizer-Ergebnis verdaechtig ist (Input traegt einen Negations-
# /Kritik-Marker, wurde aber trotzdem auf den *positiven* Kern-Begriff gemappt).
NEGATION_MARKERS = ["anti", "kritisch", "gegner", "feindlich", "skeptisch", "ablehnend", "nicht ", "kein ", "keine "]

def looks_oppositional(text: str) -> bool:
    lower = text.lower()
    return any(m in lower for m in NEGATION_MARKERS)

print(f"{len(_TECHNIQUES)} Techniken, {len(ROLE_NAMES)} Rollen, {len(STROEMUNG_NAMES)} Stroemungen als kanonische Basis")

## 2 — Testfaelle generieren

Fuer jeden kanonischen Namen werden ein paar generische Negations-/Kritik-Varianten gebildet (`anti-<name>`, `<name>-kritisch`, `nicht <name>`, `kein <name>`). Nicht jede Variante ist grammatisch elegant — es geht nur darum, ob der Normalizer sie trotzdem auf den unveraenderten Kernbegriff mappt.

In [ ]:
def gen_variants(name: str) -> list[str]:
    base = name.lower()
    return [f"anti-{base}", f"{base}-kritisch", f"nicht {base}", f"kein {base}", f"{base}gegner"]

technique_cases = [(t["name"], v) for t in _TECHNIQUES for v in gen_variants(t["name"])]
role_cases      = [(r, v) for r in ROLE_NAMES for v in gen_variants(r)]
stroemung_cases = [(s, v) for s in STROEMUNG_NAMES for v in gen_variants(s)]

print(f"{len(technique_cases)} Technik-Testfaelle, {len(role_cases)} Rollen-Testfaelle, {len(stroemung_cases)} Stroemungs-Testfaelle")

## 3 — Alle Testfaelle durchlaufen

Kann bei vielen Techniken/Stroemungen etwas dauern (ein ChromaDB-Query pro Fall).

In [ ]:
def check(cases: list[tuple[str, str]], normalize_fn, kind: str) -> pd.DataFrame:
    rows = []
    for canonical, variant in cases:
        result = normalize_fn(variant)
        collapsed = looks_oppositional(variant) and result == canonical
        rows.append({
            "kind": kind, "canonical": canonical, "variant": variant,
            "normalized_to": result, "SUSPICIOUS": collapsed,
        })
    return pd.DataFrame(rows)

df_technique = check(technique_cases, normalize_technique, "technique")
df_role      = check(role_cases, normalize_role, "role")
df_stroemung = check(stroemung_cases, normalize_stroemung, "stroemung")

df_all = pd.concat([df_technique, df_role, df_stroemung], ignore_index=True)
print(f"{len(df_all)} Faelle geprueft, {df_all['SUSPICIOUS'].sum()} davon verdaechtig")

## 4 — Verdaechtige Faelle

Jede Zeile hier ist ein moeglicher Negations-Kollaps: ein Input mit einem Kritik-/Negations-Marker, der trotzdem 1:1 auf den unveraenderten kanonischen Begriff normalisiert wurde. Sollte fuer `stroemung` leer sein (Guard aus ADR 0009) — falls `technique`/`role` hier Treffer zeigen, ist das ein neuer, bisher unentdeckter Bug.

In [ ]:
suspicious = df_all[df_all["SUSPICIOUS"]]
if suspicious.empty:
    print("Keine verdaechtigen Faelle gefunden.")
else:
    pd.set_option("display.max_rows", 200)
    display(suspicious[["kind", "canonical", "variant", "normalized_to"]])

## 5 — Zusammenfassung nach Normalizer-Typ

In [ ]:
df_all.groupby("kind")["SUSPICIOUS"].agg(["sum", "count"]).rename(
    columns={"sum": "verdaechtig", "count": "gesamt"}
)